# 锂电池热失控早期预警 - XGBoost 复现 Pipeline

**论文**: Alhasan & Alrashdan (2026), *XGBoost-Powered Predictive Analytics for Early Identification of Thermal Runaway in Lithium-Ion Batteries*, World Electric Vehicle Journal.

**说明**: 本 notebook 是干净的最终 pipeline，从头到尾线性执行能直接出 F1 和 SHAP 结果。所有探索性分析、试错、调试过程在 `01_exploration.ipynb` 里。

**与论文的差异**:
1. 过滤了初始电压 < 3.0V 的"病态电池"（论文未做，导致样本污染）
2. 标签改为 `(V<3.0 且 dV/dt<-0.1) 或 (T>80)`（论文原版 `V<3.0 | T>80` 太敏感）

---

## 工业界 notebook 的几个常见约定（写给我自己看的备忘）

- 配置（路径、超参数、种子）集中放顶部，方便修改
- 关键路径用 `assert` 提前 fail，避免下游一连串 NameError
- 中间结果缓存到 parquet，重启 kernel 不用重读 210 个 Excel
- 测试集用 `EVALUATE_TEST` 开关封存，训练阶段不许碰
- 所有 `random_state` 固定种子，保证可复现


## 0. 配置区（修改路径在这里）

In [1]:
import os
from pathlib import Path

# ==== 路径配置 ====
# 修改为你本地的实际路径
DATA_DIR = Path(os.environ.get(
    'BATTERY_DATA_DIR',
    '/Users/gtw/Desktop/新能源/锂电池/Mechanically Induced Thermal Runaway for Li-ion Batteries/excel'
))

# 缓存目录(避免每次重启 kernel 都重读 210 个 Excel)
CACHE_DIR = Path('./cache')
CACHE_DIR.mkdir(exist_ok=True)
RAW_CACHE = CACHE_DIR / 'combined_raw.parquet'

# ==== 实验配置 ====
SEED = 42  # 随机种子,固定保证可复现

# 标签阈值
VOLTAGE_THRESHOLD = 3.0     # V, 低于此电压 + 崩溃速率 -> 危险
TEMP_THRESHOLD = 80.0       # °C, 高于此温度 -> 危险
DV_CRASH_THRESHOLD = -0.1   # V/s, 电压崩溃速率(锂电池工程经验值)
SHIFT_STEPS = 5             # 标签向前平移步数(论文设定)

# XGBoost 超参数 (论文 §3.5)
XGB_PARAMS = {
    'learning_rate': 0.2,
    'max_depth': 7,
    'n_estimators': 200,
    'random_state': SEED,
    'eval_metric': 'logloss',
}

# 测试集封存开关: 训练阶段保持 False, 出最终指标时才改 True
EVALUATE_TEST = False


## 1. 数据加载（带缓存）

读 210 个 xlsx 第一次会比较慢（几分钟），所以读完缓存成 parquet。下次重启 kernel 直接读 parquet，秒级。

`assert` 是 fail-fast 习惯：路径错了立刻报错并打印实际路径，不要等到下游崩溃才找原因。


In [2]:
import pandas as pd
from tqdm.auto import tqdm

# fail-fast: 路径不对立刻报错
assert DATA_DIR.exists(), f'数据目录不存在: {DATA_DIR.resolve()}'

if RAW_CACHE.exists():
    combined_df = pd.read_parquet(RAW_CACHE)
    print(f'[load] 从缓存加载: {combined_df.shape}')
else:
    # rglob 是真正的递归搜索,不用记 recursive=True 那个坑
    files = sorted(DATA_DIR.rglob('*.xlsx'))
    assert len(files) > 0, f'未找到 xlsx 于: {DATA_DIR.resolve()}'
    print(f'[load] 找到 {len(files)} 个文件,开始读取...')
    
    dfs = []
    failed = []
    for f in tqdm(files, desc='reading'):
        try:
            df = pd.read_excel(f)
            df['file_id'] = f.name
            dfs.append(df)
        except Exception as e:
            failed.append((f.name, str(e)))
    
    if failed:
        print(f'[load] {len(failed)} 个文件读取失败,前 3 个: {failed[:3]}')
    
    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_parquet(RAW_CACHE, index=False)
    print(f'[load] 完成: {combined_df.shape}, 已缓存到 {RAW_CACHE}')

# Sanity check
print(f'文件数: {combined_df["file_id"].nunique()}')
print(f'列名: {list(combined_df.columns)}')


[load] 找到 210 个文件,开始读取...


reading:   0%|          | 0/210 [00:00<?, ?it/s]

[load] 完成: (3304693, 42), 已缓存到 cache/combined_raw.parquet
文件数: 209
列名: ['Time (second)', 'Load (lb)', 'Voltage (V)', 'Unnamed: 3', 'Unnamed: 4', 'Time (sec)', 'Penetrator Force (N)', 'Cell Voltage (V)', 'Displacement (mm)', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Time (sec) ', 'TC1 (°C)', 'TC2 (°C)', 'TC3 (°C)', 'TC4 (°C)', 'file_id', 'Test Time [s]', 'Displacement [mm]', 'Penetrator Force [mm]', 'vCell [V]', 'tAmbient [C]', 'TC1 near positive terminal [C]', 'TC2 near negative terminal [C]', 'TC3 bottom - bottom [C]', 'TC4 bottom - top [C]', 'TC5 above punch [C]', 'TC6 below punch [C]', 'Column1', 'Column2', 'Column3', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26']


## 2. 缺失值处理

按 `file_id` 分组线性插值——每块电池独立处理，避免文件 A 的最后一行被插到文件 B 的第一行。

> 工业界的"为什么":物理传感器偶尔丢点是正常的,线性插值是合理近似。但对超过 90% 缺失的列(基本是无效列),直接删掉比补出来更老实。


In [3]:
import numpy as np

# 删除缺失率 > 90% 的列(基本无效)
missing_ratio = combined_df.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.9].index.tolist()
print(f'删除高缺失列 ({len(cols_to_drop)} 列): {cols_to_drop}')
combined_df = combined_df.drop(columns=cols_to_drop)

# 按 file_id 分组线性插值
numeric_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
combined_df[numeric_cols] = (
    combined_df.groupby('file_id')[numeric_cols]
    .transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
)

print(f'清洗后: {combined_df.shape}')
remaining = combined_df.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) > 0:
    print(f'仍有缺失的列:\n{remaining}')
else:
    print('无缺失值')


删除高缺失列 (23 列): ['Unnamed: 3', 'Unnamed: 4', 'Displacement (mm)', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'TC2 (°C)', 'TC3 (°C)', 'TC4 (°C)', 'tAmbient [C]', 'Column1', 'Column2', 'Column3', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26']
清洗后: (3304693, 19)
仍有缺失的列:
Time (second)                     1489739
Load (lb)                         1489739
Voltage (V)                       1489739
Time (sec)                        1489739
Penetrator Force (N)              1489739
Cell Voltage (V)                  1489739
Time (sec)                        1489739
TC1 (°C)                          1489739
Test Time [s]                     1983473
Displacement [mm]                 1983473
Penetrator Force [mm]             1983473
vCell [V]                         1983473
TC1 near positive terminal [C]    1983473
TC2 near negative terminal [C]    1983473
TC3 bottom - bottom [C]           1983473

## 3. 特征工程

四个工程化特征（论文 §4.2）：
- `dTC1_dt` = ΔTC/Δt（温度变化率）
- `dVoltage_dt` = ΔV/Δt（电压变化率）
- `V_F_interaction` = V × F（电压-力交互项）
- `dT_V_interaction` = dTC/dt × V（温度变化率-电压交互项）

> 关键:`diff()` **必须** 在 `groupby('file_id')` 内做。否则 pandas 会把文件 A 的最后一行减到文件 B 的第一行,产生根本不存在的"跨电池变化率",这是时序数据最经典的泄露。


In [4]:
# Time 列转数值, 按 file_id + Time 排序(时序操作的前提)
combined_df['Time'] = pd.to_numeric(combined_df['Time (second)'], errors='coerce')
combined_df = combined_df.sort_values(['file_id', 'Time']).reset_index(drop=True)

# groupby 一次,后面复用
g = combined_df.groupby('file_id')

# 一阶导数(变化率)
combined_df['dTC1_dt'] = (
    g['TC1 (°C)'].transform(lambda x: x.diff()) / 
    g['Time'].transform(lambda x: x.diff())
)
combined_df['dVoltage_dt'] = (
    g['Voltage (V)'].transform(lambda x: x.diff()) / 
    g['Time'].transform(lambda x: x.diff())
)

# 交互项
combined_df['V_F_interaction'] = combined_df['Voltage (V)'] * combined_df['Penetrator Force (N)']
combined_df['dT_V_interaction'] = combined_df['dTC1_dt'] * combined_df['Voltage (V)']

# 只对新增的工程列填 0
# 不能填原始物理列(否则"0V" 0°C 这种假数据会被模型当真信号学)
new_feature_cols = ['dTC1_dt', 'dVoltage_dt', 'V_F_interaction', 'dT_V_interaction']
combined_df[new_feature_cols] = combined_df[new_feature_cols].fillna(0)

print(f'特征工程后: {combined_df.shape}')
print(f'新增列统计:\n{combined_df[new_feature_cols].describe().T[["mean", "std", "min", "max"]]}')


特征工程后: (3304693, 24)
新增列统计:
                        mean          std           min         max
dTC1_dt                  NaN          NaN          -inf         inf
dVoltage_dt              NaN          NaN          -inf         inf
V_F_interaction  -463.274284  1274.554042 -2.580152e+04  585.270276
dT_V_interaction         NaN          NaN          -inf         inf


/opt/anaconda3/envs/environment_BU520750/lib/python3.11/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/opt/anaconda3/envs/environment_BU520750/lib/python3.11/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/opt/anaconda3/envs/environment_BU520750/lib/python3.11/site-packages/numpy/_core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


## 4. 数据质量控制：过滤"病态电池"

论文里没做这一步，但实际跑会发现：超过 1/3 的电池实验**起始电压就 < 3.0V**。这些不是"从健康进入失控"的样本，而是天然就处于低压状态——把它们当作"危险"标签会污染训练集。

工业界做这种"sanity check + 过滤"的标准流程是：
1. 先**统计描述**：让数字告诉你问题的规模
2. 设定**明确的过滤准则**
3. 过滤后**再统计一遍**确认效果


In [5]:
# Step 1: 统计每个文件的初始电压
file_init = combined_df.groupby('file_id')['Voltage (V)'].first()
init_low = (file_init < VOLTAGE_THRESHOLD).sum()
print(f'初始电压 < {VOLTAGE_THRESHOLD}V 的文件: {init_low}/{len(file_init)} ({init_low/len(file_init):.1%})')

# Step 2: 过滤
healthy_files = file_init[file_init >= VOLTAGE_THRESHOLD].index
combined_df = combined_df[combined_df['file_id'].isin(healthy_files)].copy()

# Step 3: 确认效果
print(f'过滤后剩余: {combined_df["file_id"].nunique()} 个文件, {len(combined_df):,} 行')


初始电压 < 3.0V 的文件: 0/209 (0.0%)
过滤后剩余: 132 个文件, 1,814,954 行


## 5. 标签构造

复合阈值（修改自论文）：

```
Critical = (V < 3.0 且 dV/dt < -0.1) 或 (T > 80)
```

含义：电压低**且**还在加速下跌（电压崩溃区），或者温度过高，都判为危险。

`shift(-N)` 把标签往前移 N 步，让模型学习"未来 N 步会发生 critical"。**必须 groupby**，否则文件 A 的最后几行标签会被文件 B 的开头填充。


In [6]:
# 复合标签
combined_df['Critical_Raw'] = (
    ((combined_df['Voltage (V)'] < VOLTAGE_THRESHOLD) & 
     (combined_df['dVoltage_dt'] < DV_CRASH_THRESHOLD)) |
    (combined_df['TC1 (°C)'] > TEMP_THRESHOLD)
).astype(int)

# 向前平移 (按 file_id 分组,防止跨文件污染)
combined_df['Critical_Shifted'] = (
    combined_df.groupby('file_id')['Critical_Raw'].shift(-SHIFT_STEPS)
)

# 末尾 N 行的 Critical_Shifted 是 NaN(没有未来),删除
combined_df = combined_df.dropna(subset=['Critical_Shifted']).copy()
combined_df['Critical_Shifted'] = combined_df['Critical_Shifted'].astype(int)

# 标签分布
counts = combined_df['Critical_Shifted'].value_counts()
print(f'标签分布:')
print(f'  Normal   (0): {counts[0]:>10,} ({counts[0]/len(combined_df):.1%})')
print(f'  Critical (1): {counts[1]:>10,} ({counts[1]/len(combined_df):.1%})')


标签分布:
  Normal   (0):  1,687,800 (93.0%)
  Critical (1):    126,494 (7.0%)


## 6. 划分训练集 / 测试集（File-level）

**绝对不能用 `sklearn.train_test_split`**——那是按行随机切，同一块电池的数据会同时出现在训练集和测试集，等于让模型"提前看过答案"。

正确做法是按 `file_id` 整块分配：训练集和测试集的文件 ID 没有任何交集。

> `np.random.default_rng(seed)` 是新版 numpy 推荐的写法,比老的 `np.random.seed` 更安全（不会污染全局状态）。


In [7]:
# File-level shuffle (按文件分块, 防止时序泄露)
all_files = combined_df['file_id'].unique()
rng = np.random.default_rng(SEED)
all_files_shuffled = rng.permutation(all_files)

train_size = int(len(all_files_shuffled) * 0.8)
train_ids = all_files_shuffled[:train_size]
test_ids = all_files_shuffled[train_size:]

# Sanity check: 零交集
assert len(set(train_ids) & set(test_ids)) == 0, 'train/test 文件有重叠!'

train_df = combined_df[combined_df['file_id'].isin(train_ids)].copy()
test_df = combined_df[combined_df['file_id'].isin(test_ids)].copy()

print(f'训练集: {len(train_ids):>3} 个文件, {len(train_df):>10,} 行')
print(f'测试集: {len(test_ids):>3} 个文件, {len(test_df):>10,} 行')


训练集: 105 个文件,  1,417,479 行
测试集:  27 个文件,    396,815 行


## 7. 准备 X / Y

`clean_features` 函数处理三件事：
1. `inf` / `-inf` → `NaN` → `0`（除以 0 产生的极端值）
2. 极端数值裁剪（防止 XGBoost 数值不稳定）
3. 不修改原 `df`（用 `.copy()` 隔离）

> 把数据清洗封装成函数,而不是在 train/test 各写一遍重复代码——这是新手转工业界最该养成的习惯之一。


In [8]:
FEATURE_COLS = [
    'Voltage (V)', 'dVoltage_dt', 
    'Penetrator Force (N)', 'dTC1_dt',
    'V_F_interaction', 'dT_V_interaction'
]

def clean_features(df, cols):
    """特征矩阵清洗: 处理 inf/nan/极端值. 不修改原 df."""
    X = df[cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.clip(lower=-1e10, upper=1e10)
    return X

X_train = clean_features(train_df, FEATURE_COLS)
Y_train = train_df['Critical_Shifted']

X_test = clean_features(test_df, FEATURE_COLS)
Y_test = test_df['Critical_Shifted']

print(f'X_train: {X_train.shape}, Y_train: {Y_train.shape}')
print(f'X_test:  {X_test.shape}, Y_test:  {Y_test.shape}')
print(f'\n训练集标签: {Y_train.value_counts().to_dict()}')
print(f'测试集标签: {Y_test.value_counts().to_dict()}')


X_train: (1417479, 6), Y_train: (1417479,)
X_test:  (396815, 6), Y_test:  (396815,)

训练集标签: {0: 1316455, 1: 101024}
测试集标签: {0: 371345, 1: 25470}


## 8. 训练 XGBoost

超参数来自论文 §3.5。`time` 模块用来打印训练耗时——工业部署很在意这个。


In [9]:
import xgboost as xgb
import time

print('开始训练 XGBoost...')
t0 = time.time()

model = xgb.XGBClassifier(**XGB_PARAMS)
model.fit(X_train, Y_train)

print(f'训练耗时: {time.time()-t0:.1f} 秒')
print(f'模型规模: {model.n_estimators} 棵树, max_depth={model.max_depth}')


开始训练 XGBoost...
训练耗时: 1.8 秒
模型规模: 200 棵树, max_depth=7


## 9. 测试集评估（F1 / Precision / Recall）

**`EVALUATE_TEST` 开关**：训练调参期间保持 `False`，封存测试集；最终结果出来前一次性改成 `True`，跑完不再调模型。

论文 Table 2 报的指标：Precision = 0.99, Recall = 0.98, **F1 = 0.98**


In [10]:
if EVALUATE_TEST:
    from sklearn.metrics import (
        classification_report, confusion_matrix, f1_score, 
        ConfusionMatrixDisplay
    )
    import matplotlib.pyplot as plt
    
    # 预测
    Y_pred = model.predict(X_test)
    Y_proba = model.predict_proba(X_test)[:, 1]  # 留着给 SHAP 或 PR 曲线用
    
    # 论文 Table 2 核心指标
    print('=== Test Set Performance ===\n')
    print(classification_report(
        Y_test, Y_pred, 
        target_names=['Normal', 'Critical'], 
        digits=3
    ))
    
    f1 = f1_score(Y_test, Y_pred)
    print(f'F1 (Critical class): {f1:.3f}   [paper reports 0.98]')
    
    # 混淆矩阵
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(Y_test, Y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, 
        display_labels=['Normal', 'Critical']
    )
    disp.plot(cmap='Blues', ax=ax, values_format='d')
    ax.set_title('Confusion Matrix (Test Set, file-level split)')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('EVALUATE_TEST = False, 测试集封存中.')
    print('训练满意后, 把 Cell [配置区] 里的 EVALUATE_TEST 改成 True 再跑.')


EVALUATE_TEST = False, 测试集封存中.
训练满意后, 把 Cell [配置区] 里的 EVALUATE_TEST 改成 True 再跑.


## 10. SHAP 解释（论文 Figures 3-5）

SHAP 把模型的每个预测拆分成"每个特征贡献了多少"，是当前可解释性的事实标准。

为什么用 `TreeExplainer`：树模型专用，精确解（不是近似），快得多。

为什么采样 5000 行而不是全跑：73 万行全跑要十几分钟，子样本能得到几乎一样的全局解释。论文也是这么做的。


In [ ]:
if EVALUATE_TEST:
    import shap
    import matplotlib.pyplot as plt
    
    # TreeExplainer 是树模型专用的精确解释器
    explainer = shap.TreeExplainer(model)
    
    # 子采样
    sample_size = min(5000, len(X_test))
    X_sample = X_test.sample(n=sample_size, random_state=SEED)
    
    print(f'计算 SHAP values (n={sample_size})...')
    t0 = time.time()
    shap_values = explainer.shap_values(X_sample)
    print(f'耗时: {time.time()-t0:.1f} 秒')
    
    # === 论文 Figure 3: Summary plot (全局特征重要性) ===
    plt.figure()
    shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, show=False)
    plt.tight_layout()
    plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # === 论文 Figure 4: Voltage 的 dependence plot ===
    plt.figure()
    shap.dependence_plot(
        'Voltage (V)', shap_values, X_sample, 
        feature_names=FEATURE_COLS, show=False
    )
    plt.tight_layout()
    plt.savefig('shap_voltage_dependence.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # === 论文 Figure 5: Force 的 dependence plot ===
    plt.figure()
    shap.dependence_plot(
        'Penetrator Force (N)', shap_values, X_sample,
        feature_names=FEATURE_COLS, show=False
    )
    plt.tight_layout()
    plt.savefig('shap_force_dependence.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('EVALUATE_TEST = False, 跳过 SHAP.')


## 11. 接下来可以做的事（如果想再深入）

按由易到难排：

1. **跑 PR 曲线和 ROC 曲线**：F1 是单点指标，PR/ROC 能告诉你阈值调整空间。`from sklearn.metrics import precision_recall_curve, roc_curve`
2. **做 lead time 分析**（论文 §4.8）：对每个 critical 事件，回溯模型第一次预测为 1 的时间点，算"提前多少秒预警"。论文报 8.3 ± 3.2 秒。
3. **K-fold 交叉验证**：用 `GroupKFold` 按 `file_id` 分组，比单次 8/2 划分更可信。论文 §4.12 自己也承认这是 limitation。
4. **统一采样率后再 shift**：论文承认采样率从 0.045s 到几秒不等，所以 `shift(-5)` 在不同文件含义不同。先 resample 到 1 Hz 再 shift 才是真正的"5 秒提前预警"。

如果是为了找新能源相关工作，第 2 和第 4 点是最能体现"我读懂了论文的局限并改进了"的差异化点，简历上能写。
